In [1]:
import random, numpy, torch

SEED = 42
random.seed(SEED)
numpy.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

from mircdatasethandler import data_utils

root_folder = "../mirc-dataset/annotations/single-person/"
dataloaders = data_utils.create_dataloaders_pipeline(root_folder, k=5, batch_size=4)

/home/iago-rodrigues/miniconda3/envs/prod/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Getting masks from disk...


100%|██████████| 14368/14368 [00:03<00:00, 3900.69it/s]


Rearrangement for data for cross validation...
Getting dataloaders...


100%|██████████| 5/5 [00:00<00:00, 320.14it/s]


In [2]:
from elmprune import ELMImportanceProcessor, ImportanceProcessorConfig
from mircdatasethandler import models_utils

model = models_utils.get_model("Unet", "resnet34")
dataloader = dataloaders[0][1]

#elm_importance_processor = ELMImportanceProcessor(ImportanceProcessorConfig(), model, dataloader)

In [3]:
#importances = elm_importance_processor.compute_elm_layerwise_importances()

In [4]:
#importances = elm_importance_processor.compute_elm_layerwise_importances()

In [5]:
#importances = elm_importance_processor.compute_elm_global_importances()

In [6]:
from elmprune.utils import load_dict
from pathlib import Path
importances = load_dict(Path("test_dict.json"))


In [7]:
from elmprune import PruneProcessor




In [8]:

result = {}

In [9]:
for p in [0.2, 0.4, 0.6, 0.8]:
    prune_processor = PruneProcessor(model, importances, p, dataloader)
    prunned_model = prune_processor.execute()
    result[f"{p}"] = sum(p.numel() for p in prunned_model.parameters()) 

Selected 1888 / 9440 candidate channels (20.00%).
Pruned decoder.blocks.2.conv2.0: 1 / 64
Pruned decoder.blocks.2.conv1.0: 1 / 64
Pruned decoder.blocks.1.conv2.0: 5 / 128
Pruned decoder.blocks.1.conv1.0: 14 / 128
Pruned encoder.layer2.0.conv1: 2 / 128
Pruned decoder.blocks.0.conv2.0: 64 / 256
Pruned decoder.blocks.0.conv1.0: 54 / 256
Pruned encoder.layer3.0.downsample.0: 16 / 256
Pruned encoder.layer3.0.conv1: 24 / 256
Pruned encoder.layer3.1.conv1: 24 / 256
Pruned encoder.layer3.2.conv1: 17 / 256
Pruned encoder.layer3.3.conv1: 33 / 256
Pruned encoder.layer3.4.conv1: 31 / 256
Pruned encoder.layer3.5.conv1: 27 / 256
Pruned encoder.layer4.0.downsample.0: 181 / 512
Pruned encoder.layer4.0.conv1: 165 / 512
Pruned encoder.layer4.1.conv1: 204 / 512
Pruned encoder.layer4.2.conv1: 184 / 512
Selected 3776 / 9440 candidate channels (40.00%).
Pruned decoder.blocks.2.conv2.0: 1 / 64
Pruned decoder.blocks.2.conv1.0: 2 / 64
Pruned decoder.blocks.1.conv2.0: 17 / 128
Pruned decoder.blocks.1.conv1.0: 3

In [10]:
print(f"percentage 1 - {sum(p.numel() for p in model.parameters())} params")
for p in result.keys():
    print(f"percentage {p} - {result[p]} params")

percentage 1 - 24436659 params
percentage 0.2 - 14803850 params
percentage 0.4 - 8204753 params
percentage 0.6 - 3815119 params
percentage 0.8 - 1275013 params


In [11]:
result.keys()

dict_keys(['0.2', '0.4', '0.6', '0.8'])